# Quick start

A minimal, runnable example of the `qiskit-addon-sqd` package. We use SQD to post-process samples and estimate the ground-state energy of N₂ at equilibrium geometry.

For full workflows that run on real hardware, see the tutorials on the IBM Quantum Platform:

* [Sample-based quantum diagonalization](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization): chemistry Hamiltonian workflow
* [Sample-based Krylov quantum diagonalization](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-krylov-quantum-diagonalization): fermionic lattice model workflow

## Prepare inputs

SQD needs two inputs:

1. molecular data (integrals, electron counts)
2. bitstrings from sampling a circuit ansatz

The integrals are stored in [FCIDUMP](https://github.com/qiskit-community/qiskit-addon-sqd/blob/main/docs/tutorials/n2_fcidump_file.dat) format. The cell below generates them locally with PySCF when the file is missing, then reads it back, so the notebook stays self-contained with no download. The system is N₂ at its equilibrium bond length in the minimal STO-3G basis with the two core orbitals frozen. That leaves 8 spatial orbitals and 10 electrons, a 16-qubit problem small enough that we can also compute the exact (FCI) energy for reference.

In [1]:
import os

from pyscf import ao2mo, gto, mcscf, scf, tools

# Regenerate the FCIDUMP with PySCF when the file is missing, so the notebook
# stays self-contained with no download. N2 at equilibrium in the minimal STO-3G
# basis with the two 1s core orbitals frozen leaves 8 spatial orbitals and
# 10 active electrons, a 16-qubit problem.
fcidump_path = "n2_sto3g_fcidump_file.dat"
if not os.path.exists(fcidump_path):
    mol = gto.M(atom="N 0 0 0; N 0 0 1.09768", basis="sto-3g", spin=0, charge=0, verbose=0)
    mf = scf.RHF(mol).run()
    cas = mcscf.CASCI(mf, ncas=8, nelecas=10)  # freeze the 2 lowest (core) orbitals
    h1e, ecore = cas.get_h1eff()
    h2e = cas.get_h2eff()
    tools.fcidump.from_integrals(fcidump_path, h1e, h2e, cas.ncas, cas.nelecas, nuc=ecore, ms=0)

# Read molecular integrals from FCIDUMP file
fcidump_data = tools.fcidump.read(fcidump_path)

hcore = fcidump_data["H1"]  # One-electron integrals
eri = fcidump_data["H2"]  # Two-electron integrals
nuclear_repulsion_energy = fcidump_data["ECORE"]  # Core energy (nuclear repulsion + frozen core)
num_orbitals = fcidump_data["NORB"]  # Number of spatial orbitals

# fcidump.read hands back the two-electron integrals in PySCF's symmetry-packed
# form, shape (npair, npair). diagonalize_fermionic_hamiltonian wants the full
# 4-D tensor (norb, norb, norb, norb), so restore it.
eri = ao2mo.restore(1, eri, int(num_orbitals))
n_electrons = fcidump_data["NELEC"]  # Total number of (active) electrons
spin = fcidump_data["MS2"]  # 2 * spin quantum number

n_alpha = (n_electrons + spin) // 2
n_beta = n_electrons - n_alpha
nelec = (n_alpha, n_beta)

Parsing n2_sto3g_fcidump_file.dat


In a real workflow you would design an ansatz circuit, optimize it for hardware, and sample it. Here we skip that and draw 10,000 uniform random bitstrings instead, so the post-processing step stands on its own. This stands in for measurement data from a 16-qubit circuit.

In [2]:
import numpy as np
from qiskit_addon_sqd.counts import generate_bit_array_uniform

rng = np.random.default_rng(24)
bit_array = generate_bit_array_uniform(10_000, num_orbitals * 2, rand_seed=rng)

## Post-process with SQD

We run SQD with [`diagonalize_fermionic_hamiltonian`](https://qiskit.github.io/qiskit-addon-sqd/apidocs/qiskit_addon_sqd.fermion.html#qiskit_addon_sqd.fermion.diagonalize_fermionic_hamiltonian), which projects the Hamiltonian into subspaces spanned by the sampled bitstrings and diagonalizes it to estimate the ground-state energy.

The process is iterative. Each round samples a batch of bitstrings, diagonalizes the Hamiltonian in that subspace, and uses the resulting orbital occupancies to bias the next batch. This is the **configuration recovery** loop. The number of rounds depends mostly on `samples_per_batch` (smaller batches build up the subspace more gradually), together with `occupancies_tol` and `max_iterations`.

In [3]:
from functools import partial

from pyscf import fci
from qiskit_addon_sqd.fermion import SCIResult, diagonalize_fermionic_hamiltonian, solve_sci_batch

# Exact reference: full configuration interaction in this active space.
# Cheap here because the STO-3G active space is small (16 qubits).
exact_energy, _ = fci.direct_spin1.kernel(hcore, eri, int(num_orbitals), nelec)
exact_energy += nuclear_repulsion_energy
print(f"Exact (FCI) reference energy: {exact_energy:.6f} Ha\n")

# Configure the eigensolver
sci_solver = partial(solve_sci_batch, spin_sq=0.0, max_cycle=200)

# Track intermediate results via callback
result_history = []


def callback(results: list[SCIResult]):
    result_history.append(results)
    iteration = len(result_history)
    print(f"Iteration {iteration}")
    for i, result in enumerate(results):
        error = result.energy + nuclear_repulsion_energy - exact_energy
        print(f"  Subsample {i}")
        print(f"    Energy: {result.energy + nuclear_repulsion_energy:.6f}")
        print(f"    Subspace dimension: {np.prod(result.sci_state.amplitudes.shape)}")
        print(f"    Error vs exact: {error:.6f} Ha")


# samples_per_batch controls the number of bitstrings
# kept per subspace each round. Bigger values
# give a more complete subspace that reaches the energy
# in fewer rounds but costs more memory and time per
# diagonalization. Smaller values build the subspace up
# gradually over more rounds, and if set too low they
# plateau above the exact energy.
samples_per_batch = 60

# Convergence knobs:
#   samples_per_batch: smaller subspaces, so configuration recovery needs more
#                      rounds to build up the important determinants.
#   occupancies_tol:   tighter tolerance keeps the loop from stopping early.
#   max_iterations:    hard cap on the number of recovery rounds.
result = diagonalize_fermionic_hamiltonian(
    hcore,
    eri,
    bit_array,
    samples_per_batch=samples_per_batch,
    norb=num_orbitals,
    nelec=nelec,
    occupancies_tol=1e-7,
    max_iterations=30,
    sci_solver=sci_solver,
    symmetrize_spin=True,
    callback=callback,
    seed=np.random.default_rng(24),
)

Exact (FCI) reference energy: -107.652521 Ha

Iteration 1
  Subsample 0
    Energy: -106.952146
    Subspace dimension: 2500
    Error vs exact: 0.700375 Ha
Iteration 2
  Subsample 0
    Energy: -107.615469
    Subspace dimension: 3025
    Error vs exact: 0.037052 Ha
Iteration 3
  Subsample 0
    Energy: -107.652521
    Subspace dimension: 3136
    Error vs exact: 0.000000 Ha
Iteration 4
  Subsample 0
    Energy: -107.652521
    Subspace dimension: 3136
    Error vs exact: 0.000000 Ha


Representative output. The first line is the exact FCI reference, and the per-iteration energies fall toward it as the subspace grows. The integrals are generated locally and the subsamples are random (though seeded), so your exact numbers will differ slightly:

```
Exact (FCI) reference energy: -107.6xxxxx Ha   # printed by the notebook

Iteration 1
  Subsample 0
    Energy: -107.4xxxxx
    Subspace dimension: ...
    Error vs exact: 0.2xxxxx Ha
...
```

SQD is variational, so the estimate sits at or above the exact FCI energy and approaches it from above as the sampled subspace fills out the full active space.

## Next steps

This quickstart ran SQD's post-processing in isolation on synthetic samples. For end-to-end workflows or optimizing batch size, see the following content:

* [Sample-based quantum diagonalization](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-quantum-diagonalization): SQD on a chemistry Hamiltonian
* [Sample-based Krylov quantum diagonalization](https://quantum.cloud.ibm.com/docs/en/tutorials/sample-based-krylov-quantum-diagonalization): SQD on a fermionic lattice model
* The how-to [Bounding the subspace dimension](https://github.com/Qiskit/qiskit-addon-sqd/blob/main/docs/how_tos/choose_subspace_dimension.ipynb): sweeps `samples_per_batch` and plots the resulting error.